# Aula 12: Encadeamento Separado e Aplicações de Tabelas Hash

**Objetivo:** Aprender a estratégia de resolução de colisões por **Encadeamento Separado**, compará-la com o Endereçamento Aberto, e entender uma das aplicações mais importantes das Tabelas Hash: as Tabelas de Símbolos.

### **Parte 1: Revisão e Introdução ao Encadeamento Separado**

#### **1. Revisão do Problema da Colisão**

Na aula anterior, vimos que uma colisão ocorre quando duas chaves diferentes geram o mesmo índice de hash. A estratégia de **Endereçamento Aberto** resolve isso procurando o próximo *slot* vazio dentro do mesmo array. Isso pode levar a problemas de *clustering* e à necessidade de redimensionar e fazer *rehashing* de toda a tabela.

#### **2. Encadeamento Separado (Separate Chaining)**

O **Encadeamento Separado** oferece uma abordagem diferente.Em vez de procurar por um novo slot, cada slot da tabela hash funciona como um "balde" (*bucket*) que pode conter múltiplos itens. 

Geralmente, cada slot aponta para uma **lista encadeada**. Quando ocorre uma colisão, o novo item é simplesmente adicionado ao final da lista encadeada naquele slot específico.

*Veja Figura 8.11: Múltiplos itens com o mesmo hash são armazenados em uma lista no mesmo slot.*

**Vantagens:**
* Elimina os problemas de clustering do endereçamento aberto.
* O fator de carga pode ser maior que 1, já que cada slot pode conter múltiplos itens.
* A operação de deleção é mais simples de implementar.

**Desvantagem:**
* No pior caso, se muitas chaves colidirem no mesmo slot, a busca naquela lista encadeada se degrada para uma busca linear de complexidade **O(n)**. 

### **Parte 2: Implementação com Encadeamento Separado**

Para implementar esta versão, primeiro precisamos de uma estrutura de `No` e `ListaEncadeada` para usar em cada slot da nossa tabela.

#### **1. Estruturas Auxiliares: Nó e Lista Encadeada**


In [1]:
class No:
    """Nó para a lista encadeada, armazena o par {chave-valor}."""
    def __init__(self, chave=None, valor=None):
        self.chave = chave
        self.valor = valor
        self.next = None

class ListaEncadeadaSimples:
    """Lista encadeada para usar em cada slot da tabela hash."""
    def __init__(self):
        self.head = None
        self.tail = None

    def append(self, chave, valor):
        """Adiciona um novo nó ao final da lista."""
        novo_no = No(chave, valor)
        if self.tail:
            self.tail.next = novo_no
            self.tail = novo_no
        else:
            self.head = novo_no
            self.tail = novo_no
            
    def search(self, chave):
        """Busca um valor pela chave na lista."""
        no_atual = self.head
        while no_atual:
            if no_atual.chave == chave:
                return no_atual.valor
            no_atual = no_atual.next
        return None


#### **2. Implementação da Tabela Hash com Encadeamento**

Agora, a classe principal:


In [2]:
class TabelaHashEncadeada:
    def __init__(self):
        self.tamanho = 256
        # Cada slot agora contém uma instância de ListaEncadeadaSimples
        self.slots = [ListaEncadeadaSimples() for i in range(self.tamanho)]

    def _hash(self, chave):
        """A mesma função de hashing da aula anterior."""
        mult = 1
        valor_hash = 0
        for char in chave:
            valor_hash += mult * ord(char)
            mult += 1
        return valor_hash % self.tamanho

    def put(self, chave, valor):
        """Insere o par {chave-valor} na tabela."""
        h = self._hash(chave)
        # Adiciona o item à lista encadeada do slot correspondente
        self.slots[h].append(chave, valor)

    def get(self, chave):
        """Recupera o valor correspondente a uma chave."""
        h = self._hash(chave)
        # Busca a chave na lista encadeada do slot correspondente
        return self.slots[h].search(chave)


In [3]:
# -- Testando a implementação --
tabela_encadeada = TabelaHashEncadeada()
tabela_encadeada.put("good", "eggs")
tabela_encadeada.put("better", "ham")
tabela_encadeada.put("ad", "do not") # Chave que pode colidir
tabela_encadeada.put("ga", "collide") # Chave que pode colidir

print(f"Valor para 'good': {tabela_encadeada.get('good')}")
print(f"Valor para 'ad': {tabela_encadeada.get('ad')}")
print(f"Valor para 'ga': {tabela_encadeada.get('ga')}")
print(f"Valor para 'worst': {tabela_encadeada.get('worst')}")

Valor para 'good': eggs
Valor para 'ad': do not
Valor para 'ga': collide
Valor para 'worst': None



### **Parte 3: Análise e Aplicações Práticas**

#### **1. Endereçamento Aberto vs. Encadeamento Separado**

| Característica | Endereçamento Aberto | Encadeamento Separado |
| :--- | :--- | :--- |
| **Armazenamento** | Todos os itens no array principal. | Usa estruturas de dados auxiliares (ex: listas). |
| **Performance** | Rápido com baixo fator de carga, mas degrada com clustering. | Performance mais consistente, mas pode degradar para O(n) em um slot sobrecarregado. |
| **Memória** | Mais eficiente em uso de memória (sem ponteiros extras). | Requer memória adicional para os ponteiros da lista encadeada. |
| **Deleção** | Complexa (não se pode simplesmente apagar um slot). | Simples (remoção padrão da lista encadeada). |
| **Fator de Carga** | Deve ser mantido baixo (\< 1). | Pode ser \> 1. |

#### **2. Aplicação Prática: Tabelas de Símbolos**

Uma das aplicações mais importantes das tabelas hash é na construção de **compiladores e interpretadores**. 

  * **O que é uma Tabela de Símbolos?** É uma estrutura de dados usada pelo compilador para armazenar informações sobre os "símbolos" de um programa, como nomes de variáveis, funções e classes. 
  * **Por que usar uma Tabela Hash?** O compilador precisa acessar as informações de um símbolo (ex: o tipo de uma variável) de forma extremamente rápida. Uma tabela hash é perfeita para isso. A chave é o nome do símbolo (ex: a string `"minha_variavel"`) e o valor é um objeto contendo todas as informações sobre ele (tipo, escopo, endereço de memória, etc.). 





# Exemplo conceitual de código
nome = "Joe"
idade = 27

def saudacao():
    print("Olá!")
```

O compilador criaria uma tabela de símbolos (usando uma tabela hash) com entradas como:

  * `"nome"` -\> `{tipo: string, valor: "Joe", ...}`
  * `"idade"` -\> `{tipo: int, valor: 27, ...}`
  * `"saudacao"` -\> `{tipo: function, endereço: 0x..., ...}`

Isso permite que o compilador verifique rapidamente se uma variável foi declarada, qual seu tipo, etc., durante o processo de tradução do código. 

### **Exercícios da Aula 12**

1.  Qual é o tempo de busca no pior caso de uma tabela hash que usa a técnica de encadeamento separado? Explique o cenário em que isso ocorre. 
2.  Assumindo uma distribuição uniforme de chaves, qual é a complexidade de tempo esperada para as operações de Inserção, Deleção e Busca em uma tabela hash? 
3.  No nosso exemplo de implementação da `TabelaHashEncadeada`, a lista encadeada não trata o caso de chaves duplicadas (ou seja, se você fizer `put('a', 1)` e depois `put('a', 2)`, ambos estarão na lista). Modifique o método `append` da `ListaEncadeadaSimples` para que, se a chave já existir, ele atualize o valor em vez de adicionar um novo nó.